# Training BiLSTM for NER (PII Detection)

- **Model**: `BiLSTM` with VnCoreNLP word segmentation
- **Data**: `quynong/cs419-data` from HuggingFace
- **Evaluation**:
  1. **Classification**: Binary (has entity vs no entity)
  2. **NER**: Micro Precision / Recall / F1 at entity level

In [1]:
!pip install -q transformers datasets seqeval accelerate vncorenlp py_vncorenlp

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 3.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 23.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 37.8 MB/s eta 0:00:00


In [2]:
import json
import numpy as np
import torch
from pathlib import Path
from datasets import load_dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification,
)
from seqeval.metrics import (
    precision_score,
    recall_score,
    f1_score,
    classification_report,
)
from sklearn.metrics import precision_recall_fscore_support, accuracy_score

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

Device: cuda


## 1. Load Dataset from HuggingFace

In [10]:
dataset = load_dataset("quynong/cs419-data")
print(dataset)
print(f"\nTrain sample:")
print(dataset['train'][0])

README.md:   0%|          | 0.00/620 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/9.76M [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/1.09M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/54117 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/6014 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['source_text', 'language', 'privacy_mask'],
        num_rows: 54117
    })
    validation: Dataset({
        features: ['source_text', 'language', 'privacy_mask'],
        num_rows: 6014
    })
})

Train sample:
{'source_text': 'Chúng tôi có những lo ngại về một số giao dịch từ tài khoản 0041000990011 (Vietcombank), IBAN PK51MGLA0900120022020017. Vui lòng kiểm tra nhật ký từ 08/05/2022 đến Ngày 2/2/1951.', 'language': 'vi', 'privacy_mask': [{'start': 60, 'end': 87, 'label': 'ACCOUNTNUMBER', 'value': '0041000990011 (Vietcombank)'}, {'start': 94, 'end': 118, 'label': 'IBAN', 'value': 'PK51MGLA0900120022020017'}, {'start': 149, 'end': 159, 'label': 'DATE', 'value': '08/05/2022'}, {'start': 164, 'end': 177, 'label': 'DATE', 'value': 'Ngày 2/2/1951'}]}


## 2. Setup PhoBERT Tokenizer & Word Segmentation

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [19]:
import py_vncorenlp

# Download VnCoreNLP models (only need to run once)
py_vncorenlp.download_model(save_dir='/content/drive/MyDrive/CS419')
segmenter = py_vncorenlp.VnCoreNLP(save_dir='/content/drive/MyDrive/CS419')

MODEL_NAME = "vinai/phobert-base"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print(f"Tokenizer: {MODEL_NAME}")
print(f"Vocab size: {tokenizer.vocab_size}")

VnCoreNLP model folder /content/drive/MyDrive/CS419 already exists! Please load VnCoreNLP from this folder!


config.json:   0%|          | 0.00/557 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

bpe.codes: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Tokenizer: vinai/phobert-base
Vocab size: 64000


## 3. Build Label Set from Data

In [11]:
# Collect all unique entity labels
all_labels = set()
for split in dataset:
    for sample in dataset[split]:
        for ent in sample['privacy_mask']:
            all_labels.add(ent['label'])

all_labels = sorted(all_labels)
print(f"Found {len(all_labels)} entity types:")
print(all_labels)

# Build BIO label list
label_list = ["O"]
for lbl in all_labels:
    label_list.append(f"B-{lbl}")
    label_list.append(f"I-{lbl}")

label2id = {l: i for i, l in enumerate(label_list)}
id2label = {i: l for i, l in enumerate(label_list)}
num_labels = len(label_list)

print(f"\nTotal BIO labels: {num_labels}")
print(f"First 10: {label_list[:10]}")

Found 54 entity types:
['ACCOUNTNAME', 'ACCOUNTNUMBER', 'AGE', 'AMOUNT', 'BIC', 'BITCOINADDRESS', 'BUILDINGNUMBER', 'CCCD', 'CITY', 'COMPANYNAME', 'COUNTY', 'CREDITCARDCVV', 'CREDITCARDISSUER', 'CREDITCARDNUMBER', 'CURRENCY', 'CURRENCYCODE', 'CURRENCYNAME', 'CURRENCYSYMBOL', 'DATE', 'DOB', 'EMAIL', 'ETHEREUMADDRESS', 'EYECOLOR', 'FIRSTNAME', 'GENDER', 'HEIGHT', 'IBAN', 'IPADDRESS', 'JOBAREA', 'JOBTITLE', 'JOBTYPE', 'LASTNAME', 'LITECOINADDRESS', 'MAC', 'MASKEDNUMBER', 'MIDDLENAME', 'NEARBYGPSCOORDINATE', 'ORDINALDIRECTION', 'PASSWORD', 'PHONEIMEI', 'PHONENUMBER', 'PIN', 'PREFIX', 'SECONDARYADDRESS', 'SEX', 'STATE', 'STREET', 'TIME', 'URL', 'USERAGENT', 'USERNAME', 'VEHICLEVIN', 'VEHICLEVRM', 'ZIPCODE']

Total BIO labels: 109
First 10: ['O', 'B-ACCOUNTNAME', 'I-ACCOUNTNAME', 'B-ACCOUNTNUMBER', 'I-ACCOUNTNUMBER', 'B-AGE', 'I-AGE', 'B-AMOUNT', 'I-AMOUNT', 'B-BIC']


## 4. Tokenize & Align Labels

For each sample:
1. Segment Vietnamese text with VnCoreNLP
2. Tokenize with PhoBERT tokenizer
3. Align character-level entity spans to subword tokens using BIO scheme

In [12]:
MAX_LENGTH = 256

def segment_text(text):
    """Word-segment Vietnamese text using VnCoreNLP."""
    try:
        sentences = segmenter.word_segment(text)
        return " ".join(sentences)
    except:
        return text


def char_to_token_labels(source_text, privacy_mask, encoding, segmented_text):
    """
    Convert character-level entity spans to token-level BIO labels.
    Uses offset_mapping from tokenizer to align.
    """
    # Build character-level label array for the ORIGINAL text
    char_labels = ['O'] * len(source_text)
    for ent in privacy_mask:
        start, end, label = ent['start'], ent['end'], ent['label']
        if start >= len(source_text) or end > len(source_text):
            continue
        char_labels[start] = f"B-{label}"
        for i in range(start + 1, end):
            char_labels[i] = f"I-{label}"

    # Map from segmented text positions back to original text positions
    # VnCoreNLP replaces spaces within words with '_' and keeps structure
    # We need a char map: segmented_pos -> original_pos
    seg_to_orig = []
    orig_idx = 0
    for seg_idx, seg_char in enumerate(segmented_text):
        if seg_char == '_' and orig_idx < len(source_text) and source_text[orig_idx] == ' ':
            seg_to_orig.append(orig_idx)
            orig_idx += 1
        elif seg_char == ' ':
            # Extra space added by segmenter between words
            # Check if original also has space
            if orig_idx < len(source_text) and source_text[orig_idx] == ' ':
                seg_to_orig.append(orig_idx)
                orig_idx += 1
            else:
                seg_to_orig.append(-1)  # no mapping
        else:
            if orig_idx < len(source_text):
                seg_to_orig.append(orig_idx)
                orig_idx += 1
            else:
                seg_to_orig.append(-1)

    # Now assign labels to each token using offset_mapping
    offset_mapping = encoding.get('offset_mapping', [])
    token_labels = []

    prev_label = 'O'
    for (tok_start, tok_end) in offset_mapping:
        if tok_start == 0 and tok_end == 0:
            # Special token
            token_labels.append(-100)
            continue

        # Get the original char positions for this token span
        orig_positions = []
        for seg_pos in range(tok_start, min(tok_end, len(seg_to_orig))):
            if seg_pos < len(seg_to_orig) and seg_to_orig[seg_pos] >= 0:
                orig_positions.append(seg_to_orig[seg_pos])

        if not orig_positions:
            token_labels.append(label2id['O'])
            prev_label = 'O'
            continue

        # Use the label of the first character of this token
        first_orig_pos = orig_positions[0]
        if first_orig_pos < len(char_labels):
            lbl = char_labels[first_orig_pos]
        else:
            lbl = 'O'

        if lbl in label2id:
            token_labels.append(label2id[lbl])
        else:
            token_labels.append(label2id['O'])
        prev_label = lbl

    return token_labels

print("Label alignment function loaded.")

Label alignment function loaded.


In [13]:
def tokenize_and_align(examples):
    """
    Tokenize batch of examples and align entity labels to subword tokens.
    """
    all_input_ids = []
    all_attention_mask = []
    all_labels = []

    for source_text, privacy_mask in zip(examples['source_text'], examples['privacy_mask']):
        # Step 1: Word segment
        segmented = segment_text(source_text)

        # Step 2: Tokenize (Bỏ return_offsets_mapping vì PhoBERT không hỗ trợ)
        encoding = tokenizer(
            segmented,
            max_length=MAX_LENGTH,
            truncation=True,
            padding='max_length',
        )

        # --- BƯỚC BỔ SUNG: TỰ TẠO OFFSET MAPPING CHO PHOBERT ---
        input_ids = encoding['input_ids']
        tokens = tokenizer.convert_ids_to_tokens(input_ids)

        offset_mapping = []
        curr_pos = 0

        for token in tokens:
            # 1. Các token đặc biệt (<s>, </s>, <pad>, <unk>) cho offset (0, 0)
            if token in tokenizer.all_special_tokens:
                offset_mapping.append((0, 0))
                continue

            # 2. Xóa bỏ ký tự '@@' đặc trưng của PhoBERT subword để lấy độ dài thực
            clean_token = token.replace("@@", "")

            # 3. Bỏ qua các khoảng trắng dư thừa trong chuỗi segmented
            while curr_pos < len(segmented) and segmented[curr_pos] == ' ':
                curr_pos += 1

            start = curr_pos
            end = curr_pos + len(clean_token)

            offset_mapping.append((start, end))
            curr_pos = end

        # Nhúng offset_mapping tự tạo vào encoding để hàm char_to_token_labels sử dụng
        encoding['offset_mapping'] = offset_mapping
        # --------------------------------------------------------

        # Step 3: Align labels (Giữ nguyên hàm của bạn, nó đã hoạt động rất tốt!)
        token_labels = char_to_token_labels(
            source_text, privacy_mask, encoding, segmented
        )

        # Pad labels to MAX_LENGTH
        while len(token_labels) < MAX_LENGTH:
            token_labels.append(-100)
        token_labels = token_labels[:MAX_LENGTH]

        all_input_ids.append(encoding['input_ids'])
        all_attention_mask.append(encoding['attention_mask'])
        all_labels.append(token_labels)

    return {
        'input_ids': all_input_ids,
        'attention_mask': all_attention_mask,
        'labels': all_labels,
    }

print("Tokenization function loaded & patched for PhoBERT.")

Tokenization function loaded & patched for PhoBERT.


In [24]:
# Apply tokenization to all splits
tokenized_dataset = dataset.map(
    tokenize_and_align,
    batched=True,
    batch_size=32,
    remove_columns=dataset['train'].column_names,
    desc="Tokenizing",
)

print(tokenized_dataset)
print(f"\nSample input_ids length: {len(tokenized_dataset['train'][0]['input_ids'])}")
print(f"Sample labels length: {len(tokenized_dataset['train'][0]['labels'])}")

Parameter 'function'=<function tokenize_and_align at 0x7ac9bc0ab740> of the transform datasets.arrow_dataset.Dataset._map_single couldn't be hashed properly, a random hash was used instead. Make sure your transforms and parameters are serializable with pickle or dill for the dataset fingerprinting and caching to work. If you reuse this transform, the caching mechanism will consider it to be different from the previous calls and recompute everything. This warning is only showed once. Subsequent hashing failures won't be showed.


Tokenizing:   0%|          | 0/54117 [00:00<?, ? examples/s]

Tokenizing:   0%|          | 0/6014 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 54117
    })
    validation: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 6014
    })
})

Sample input_ids length: 256
Sample labels length: 256


In [25]:
# Lệnh này sẽ "đóng gói" toàn bộ dữ liệu đã xử lý và gửi vào Drive
tokenized_dataset.save_to_disk('/content/drive/MyDrive/CS419/tokenized_phobert_data')
print("Đã tạo xong thư mục dữ liệu tại: /content/drive/MyDrive/CS419/tokenized_phobert_data")

Saving the dataset (0/1 shards):   0%|          | 0/54117 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/6014 [00:00<?, ? examples/s]

Đã tạo xong thư mục dữ liệu tại: /content/drive/MyDrive/CS419/tokenized_phobert_data


## 5. Define Model & Training

In [26]:
import torch
import torch.nn as nn

# Tạo một class Config giả lập để đánh lừa HuggingFace Trainer
class DummyConfig:
    def __init__(self, id2label, label2id):
        self.id2label = id2label
        self.label2id = label2id

class BiLSTM_NER(nn.Module):
    def __init__(self, vocab_size, num_labels, id2label, label2id, padding_idx=1, emb_dim=300, hidden_dim=256):
        super().__init__()
        self.num_labels = num_labels

        # Gắn config chứa id2label và label2id vào model
        self.config = DummyConfig(id2label, label2id)

        # 1. Lớp Embedding (Biến ID thành Vector) - Học từ con số 0
        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=padding_idx)

        # 2. Lớp BiLSTM (Học ngữ cảnh 2 chiều)
        self.bilstm = nn.LSTM(
            input_size=emb_dim,
            hidden_size=hidden_dim // 2,
            num_layers=2,
            bidirectional=True,
            batch_first=True,
            dropout=0.3
        )

        self.dropout = nn.Dropout(0.3)

        # 3. Lớp Phân loại (Dự đoán nhãn)
        self.classifier = nn.Linear(hidden_dim, num_labels)

        # Hàm Loss bỏ qua các token đệm (-100)
        self.loss_fct = nn.CrossEntropyLoss(ignore_index=-100)

    def forward(self, input_ids, attention_mask=None, labels=None, **kwargs):
        # Biến đổi token ID thành Vector
        embeds = self.embedding(input_ids)

        # Đưa qua BiLSTM
        lstm_out, _ = self.bilstm(embeds)
        lstm_out = self.dropout(lstm_out)

        # Tính điểm số cho từng nhãn
        logits = self.classifier(lstm_out)

        loss = None
        if labels is not None:
            # Tính CrossEntropy Loss
            loss = self.loss_fct(logits.view(-1, self.num_labels), labels.view(-1))

        # Trả về dict theo đúng chuẩn mà HuggingFace Trainer yêu cầu
        return {"loss": loss, "logits": logits} if loss is not None else {"logits": logits}

# Khởi tạo model (Lưu ý truyền thêm id2label và label2id vào)
# vocab_size của PhoBERT mặc định là 64000
model = BiLSTM_NER(
    vocab_size=tokenizer.vocab_size,
    num_labels=num_labels,
    id2label=id2label,       # Bổ sung dòng này
    label2id=label2id,       # Bổ sung dòng này
    padding_idx=tokenizer.pad_token_id
)
model.to(device)

print(f"BiLSTM Model initialized with {num_labels} labels")
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

BiLSTM Model initialized with 109 labels
Parameters: 20,063,597


In [27]:
# Cần dùng DataCollator mặc định cho PyTorch vì đây là custom model
from transformers import DefaultDataCollator
data_collator = DefaultDataCollator()

training_args = TrainingArguments(
    output_dir="./bilstm-ner-pii",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-3,          # <-- Tăng LR lên 2e-3 (PhoBERT là 5e-5)
    per_device_train_batch_size=32, # Batch size lớn hơn một chút cho LSTM
    per_device_eval_batch_size=64,
    num_train_epochs=15,         # <-- Tăng lên 15 Epochs vì học từ đầu
    weight_decay=1e-4,
    logging_steps=100,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    fp16=torch.cuda.is_available(),
    report_to="none",
)

print("Training arguments configured for BiLSTM.")

Training arguments configured for BiLSTM.


## 6. Evaluation Metrics

Two evaluation modes:
1. **NER (Entity-level)**: seqeval micro P/R/F1
2. **Classification (Sentence-level)**: Does the sentence contain any entity?

In [29]:
def compute_metrics(eval_preds):
    """
    Compute both NER entity-level and sentence-level classification metrics.
    """
    logits, labels = eval_preds
    predictions = np.argmax(logits, axis=-1)

    # ──── NER Entity-level metrics (seqeval) ────────────────────────────────
    true_labels = []
    true_predictions = []

    for pred_seq, label_seq in zip(predictions, labels):
        pred_tags = []
        true_tags = []
        for p, l in zip(pred_seq, label_seq):
            if l == -100:
                continue
            pred_tags.append(id2label[p])
            true_tags.append(id2label[l])
        true_predictions.append(pred_tags)
        true_labels.append(true_tags)

    ner_precision = precision_score(true_labels, true_predictions, average='micro')
    ner_recall = recall_score(true_labels, true_predictions, average='micro')
    ner_f1 = f1_score(true_labels, true_predictions, average='micro')

    # ──── Classification: sentence has entity or not ────────────────────────
    cls_true = []
    cls_pred = []

    for pred_seq, label_seq in zip(predictions, labels):
        # True: does this sentence have any non-O label?
        has_entity_true = any(
            l not in (-100, label2id['O']) for l in label_seq
        )
        # Pred: does prediction contain any non-O?
        has_entity_pred = any(
            p != label2id['O'] for p, l in zip(pred_seq, label_seq) if l != -100
        )
        cls_true.append(int(has_entity_true))
        cls_pred.append(int(has_entity_pred))

    cls_precision, cls_recall, cls_f1, _ = precision_recall_fscore_support(
        cls_true, cls_pred, average='binary', zero_division=0
    )
    cls_accuracy = accuracy_score(cls_true, cls_pred)

    return {
        # NER entity-level
        "precision": ner_precision,
        "recall": ner_recall,
        "f1": ner_f1,
        # Classification sentence-level
        "cls_accuracy": cls_accuracy,
        "cls_precision": cls_precision,
        "cls_recall": cls_recall,
        "cls_f1": cls_f1,
    }

print("Metrics function loaded.")

Metrics function loaded.


In [30]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset['train'],
    eval_dataset=tokenized_dataset['validation'],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    # Lưu ý: Bỏ thuộc tính processing_class=tokenizer vì Trainer không cần cho custom nn.Module
)

print("Trainer ready. Starting training BiLSTM...")
trainer.train()

Trainer ready. Starting training BiLSTM...


Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Cls Accuracy,Cls Precision,Cls Recall,Cls F1
1,0.083640,0.060909,0.849710,0.875772,0.862544,0.997838,0.997325,0.998810,0.998067
2,0.059637,0.055091,0.859245,0.895544,0.877019,0.998670,0.997920,0.999702,0.998811
3,0.047139,0.044511,0.883697,0.910204,0.896755,0.998503,0.997624,0.999702,0.998662
4,0.030763,0.044916,0.895634,0.922068,0.908659,0.999169,0.998514,1.000000,0.999257
5,0.037295,0.050434,0.906924,0.924769,0.915759,0.999169,0.998514,1.000000,0.999257
6,0.018362,0.051417,0.908609,0.926312,0.917375,0.998670,0.998216,0.999405,0.998810
7,0.015189,0.051264,0.906124,0.929109,0.917472,0.998503,0.997624,0.999702,0.998662
8,0.013433,0.058103,0.908406,0.929784,0.918970,0.999169,0.998514,1.000000,0.999257
9,0.009558,0.058688,0.911840,0.927758,0.919730,0.998836,0.998217,0.999702,0.998959
10,0.007743,0.062138,0.908492,0.930748,0.919485,0.998337,0.997328,0.999702,0.998514


TrainOutput(global_step=25380, training_loss=0.031888232868637414, metrics={'train_runtime': 1078.3393, 'train_samples_per_second': 752.783, 'train_steps_per_second': 23.536, 'total_flos': 0.0, 'train_loss': 0.031888232868637414, 'epoch': 15.0})

## 7. Final Evaluation

In [31]:
# Run final evaluation on validation set
eval_results = trainer.evaluate()

print("="*60)
print("FINAL EVALUATION RESULTS")
print("="*60)
print(f"\n{'─'*40}")
print("NER Entity-Level (Micro):")
print(f"  Precision: {eval_results['eval_precision']:.4f}")
print(f"  Recall:    {eval_results['eval_recall']:.4f}")
print(f"  F1:        {eval_results['eval_f1']:.4f}")
print(f"\n{'─'*40}")
print("Classification (Has Entity):")
print(f"  Accuracy:  {eval_results['eval_cls_accuracy']:.4f}")
print(f"  Precision: {eval_results['eval_cls_precision']:.4f}")
print(f"  Recall:    {eval_results['eval_cls_recall']:.4f}")
print(f"  F1:        {eval_results['eval_cls_f1']:.4f}")
print("="*60)

FINAL EVALUATION RESULTS

────────────────────────────────────────
NER Entity-Level (Micro):
  Precision: 0.9161
  Recall:    0.9329
  F1:        0.9244

────────────────────────────────────────
Classification (Has Entity):
  Accuracy:  0.9988
  Precision: 0.9982
  Recall:    0.9997
  F1:        0.9990


In [32]:
# Detailed NER report per entity type
eval_output = trainer.predict(tokenized_dataset['validation'])
predictions = np.argmax(eval_output.predictions, axis=-1)
labels = eval_output.label_ids

true_labels = []
true_predictions = []

for pred_seq, label_seq in zip(predictions, labels):
    pred_tags = []
    true_tags = []
    for p, l in zip(pred_seq, label_seq):
        if l == -100:
            continue
        pred_tags.append(id2label[p])
        true_tags.append(id2label[l])
    true_predictions.append(pred_tags)
    true_labels.append(true_tags)

report = classification_report(true_labels, true_predictions, digits=4)
print("Detailed NER Classification Report (per entity type):")
print(report)

Detailed NER Classification Report (per entity type):
                     precision    recall  f1-score   support

        ACCOUNTNAME     0.9746    0.9829    0.9787       234
      ACCOUNTNUMBER     0.9843    0.9881    0.9862       253
                AGE     0.8925    0.9540    0.9222       174
             AMOUNT     0.9378    0.9536    0.9456       237
                BIC     0.8485    0.9825    0.9106        57
     BITCOINADDRESS     0.9091    0.9497    0.9290       179
     BUILDINGNUMBER     0.9339    0.9381    0.9360       226
               CCCD     0.9184    0.9712    0.9441       139
               CITY     0.9803    0.9755    0.9779       204
        COMPANYNAME     0.8844    0.7969    0.8384       192
             COUNTY     0.9821    0.9955    0.9888       221
      CREDITCARDCVV     0.9167    0.9167    0.9167        60
   CREDITCARDISSUER     0.9259    0.9346    0.9302       107
   CREDITCARDNUMBER     0.7056    0.8424    0.7680       165
           CURRENCY     0.8957

## 8. Save Model

In [33]:
import os

save_path = "/content/drive/MyDrive/CS419/bilstm-ner-pii/best_model"
os.makedirs(save_path, exist_ok=True)

# Lưu trọng số của Model
torch.save(model.state_dict(), os.path.join(save_path, "bilstm_weights.pth"))

# Lưu Tokenizer để dùng lại lúc Inference
tokenizer.save_pretrained(save_path)

print(f"BiLSTM Model weights and Tokenizer saved to: {save_path}")

BiLSTM Model weights and Tokenizer saved to: /content/drive/MyDrive/CS419/bilstm-ner-pii/best_model


## 9. Inference Example

In [34]:
def predict_entities(text):
    """Run NER inference on a single Vietnamese text using custom BiLSTM model."""
    # 1. Tách từ bằng VnCoreNLP (Bắt buộc vì BiLSTM được train trên dữ liệu đã tách từ)
    segmented = segment_text(text)

    # 2. Tokenize (Sử dụng Tokenizer của PhoBERT)
    encoding = tokenizer(
        segmented,
        max_length=MAX_LENGTH,
        truncation=True,
        padding='max_length',
        return_tensors='pt'
    )

    # 3. Tự tạo Offset Mapping (Vì PhoBERT Tokenizer không hỗ trợ sẵn)
    input_ids = encoding['input_ids'][0].tolist()
    tokens = tokenizer.convert_ids_to_tokens(input_ids)

    offset_mapping = []
    curr_pos = 0

    for token in tokens:
        if token in tokenizer.all_special_tokens:
            offset_mapping.append((0, 0))
            continue

        clean_token = token.replace("@@", "")

        while curr_pos < len(segmented) and segmented[curr_pos] == ' ':
            curr_pos += 1

        start = curr_pos
        end = curr_pos + len(clean_token)

        offset_mapping.append((start, end))
        curr_pos = end

    # 4. Đưa dữ liệu vào thiết bị (GPU/CPU)
    # Lưu ý: BiLSTM chỉ cần input_ids, các tham số khác như attention_mask là tùy chọn
    inputs = {k: v.to(device) for k, v in encoding.items()}

    # 5. Dự đoán
    model.eval()
    with torch.no_grad():
        outputs = model(**inputs)

    # KHÁC BIỆT Ở ĐÂY: Truy cập logits bằng key ['logits'] vì model trả về dict
    logits = outputs['logits']
    preds = torch.argmax(logits, dim=-1)[0].cpu().tolist()

    # 6. Trích xuất thực thể từ BIO predictions
    entities = []
    current_entity = None

    for idx, (pred_id, (start, end)) in enumerate(zip(preds, offset_mapping)):
        if start == 0 and end == 0:
            continue

        # Dùng id2label đã khai báo ở Cell 3
        label = id2label[pred_id]

        if label.startswith('B-'):
            if current_entity:
                entities.append(current_entity)
            current_entity = {
                'label': label[2:],
                'start': start,
                'end': end,
                'text': segmented[start:end],
            }
        elif label.startswith('I-') and current_entity:
            current_entity['end'] = end
            current_entity['text'] = segmented[current_entity['start']:end]
        else:
            if current_entity:
                entities.append(current_entity)
                current_entity = None

    if current_entity:
        entities.append(current_entity)

    # 7. Làm sạch text (xóa dấu '_' để trả về text gốc giống người dùng nhập)
    for ent in entities:
        ent['text'] = ent['text'].replace('_', ' ')

    return entities

# --- CHẠY THỬ NGHIỆM ---
test_text = "Xin chào, tôi là Nguyễn Văn An, số điện thoại 0912345678, địa chỉ 123 Lê Lợi, Quận 1."
entities = predict_entities(test_text)

print(f"Text: {test_text}")
print(f"\nEntities found by BiLSTM: {len(entities)}")
for ent in entities:
    print(f"  [{ent['label']}] \"{ent['text']}\" (pos {ent['start']}-{ent['end']})")

Text: Xin chào, tôi là Nguyễn Văn An, số điện thoại 0912345678, địa chỉ 123 Lê Lợi, Quận 1.

Entities found by BiLSTM: 2
  [PHONENUMBER] "0912345678" (pos 48-58)
  [COUNTY] "Quận 1" (pos 82-88)
